In [23]:
import pandas as pd
import numpy as np
import os
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Téléchargement des ressources NLP
nltk.download('punkt')
nltk.download('stopwords')

# Chemins basés sur ton arborescence
PROCESSED_PATH = "../data/processed/"

print("✅ Environnement prêt et chemins configurés.")

✅ Environnement prêt et chemins configurés.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [24]:
# Chargement du dataset principal
file_path = os.path.join(PROCESSED_PATH, "dataset_merged_final.csv")

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ Dataset chargé avec succès : {df.shape[0]} lignes et {df.shape[1]} colonnes.")
else:
    print("⚠️ Fichier introuvable. Vérifie le nom dans data/processed/.")

df.head()

✅ Dataset chargé avec succès : 5423 lignes et 51 colonnes.


,source,report_id,age,sex,weight_kg,n_drugs,suspect_drugs,concomitant_drugs,all_drugs,avg_treatment_duration_days,...,n_traitements_anterieurs,chirurgie_recente,dialyse,medicaments_concomitants,classes_therapeutiques,n_medicaments_risque_eleve,medicaments_risque_eleve,polymedicaments_haut_risque,narrative_words,severity_label
0,VAERS_FDA,5801206-7,26.0,M,NaN,1.0,DURAGESIC-100,NaN,DURAGESIC-100,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,VAERS_FDA,10003300,77.0,F,NaN,1.0,BONIVA,NaN,BONIVA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,VAERS_FDA,10003301,NaN,F,NaN,1.0,IBUPROFEN,NaN,IBUPROFEN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,VAERS_FDA,10003311,76.0,F,NaN,4.0,LETAIRIS | LETAIRIS | LETAIRIS,TRACLEER,LETAIRIS | LETAIRIS | LETAIRIS | TRACLEER,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VAERS_FDA,10003312,43.0,F,41.8,6.0,ILARIS | ILARIS,PREDNISONE | RELPAX | ULTRACET,ILARIS | ILARIS | PREDNISONE | RELPAX | ULTRACET,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# 1. Suppression des doublons
df = df.drop_duplicates()

# 2. Gestion des valeurs manquantes
# On remplit les colonnes textuelles par 'Unknown' et les numériques par la médiane
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('Unknown')
    else:
        df[col] = df[col].fillna(df[col].median())

print("✅ Doublons supprimés et valeurs manquantes traitées.")

✅ Doublons supprimés et valeurs manquantes traitées.


c:\Users\ADMIN\Desktop\projet datascience\venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [26]:
stop_words = set(stopwords.words('english'))

def clean_medical_text(text):
    if not isinstance(text, str) or text == 'Unknown':
        return ""
    # Nettoyage : minuscules, retrait ponctuation/chiffres
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenization et retrait des stop words
    tokens = word_tokenize(text)
    filtered = [t for t in tokens if t not in stop_words]
    return " ".join(filtered)

# Applique ceci sur ta colonne de texte (ex: 'symptom_description' ou 'REPORT')
# Remplace 'SYMPTOM_TEXT' par le nom exact de ta colonne texte si nécessaire
text_col = 'SYMPTOM_TEXT' 

if text_col in df.columns:
    print("Nettoyage NLP en cours...")
    df['cleaned_text'] = df[text_col].apply(clean_medical_text)
    print("✅ Colonne texte nettoyée.")
else:
    print(f"ℹ️ Colonne '{text_col}' non trouvée, étape NLP ignorée.")

ℹ️ Colonne 'SYMPTOM_TEXT' non trouvée, étape NLP ignorée.


In [27]:
from sklearn.preprocessing import LabelEncoder

# Liste des colonnes à transformer (adapte selon tes noms de colonnes réels)
categorical_cols = ['gender', 'vaccine_type', 'medical_history', 'severity_level']

le = LabelEncoder()

for col in categorical_cols:
    if col in df.columns:
        df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
        print(f"✅ Encodage terminé pour : {col}")

print("✅ Prétraitement catégoriel terminé.")

✅ Prétraitement catégoriel terminé.


In [28]:
output_file = os.path.join(PROCESSED_PATH, "dataset_preprocessed.csv")
df.to_csv(output_file, index=False)

print(f"🚀 Notebook 02 terminé ! Fichier prêt pour le Feature Engineering : {output_file}")

🚀 Notebook 02 terminé ! Fichier prêt pour le Feature Engineering : ../data/processed/dataset_preprocessed.csv
